In [17]:
import requests
from bs4 import BeautifulSoup
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [19]:
# Step 1: Data Collection
url = "https://www.gutenberg.org/cache/epub/67979/pg67979-images.html"
response = requests.get(url)
text = response.text

def preprocess_text(text):
    cleaned_text = text.lower()
    cleaned_text = re.sub(r'[^\w\s]', '', cleaned_text)
    tokens = word_tokenize(cleaned_text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in tokens if token not in stop_words]
    return filtered_tokens

preprocessed_text = preprocess_text(text)
print(preprocessed_text)

['doctype', 'html', 'html', 'langen', 'head', 'meta', 'charsetutf8style', 'pgheader', 'div', 'pgfooter', 'div', 'initial', 'display', 'block', 'margintop', '1em', 'marginbottom', '1em', 'marginleft', '2em', 'pgfooter', 'divagate', 'fontsize', '90', 'margintop', '0', 'marginbottom', '0', 'textalign', 'center', 'pgfooter', 'li', 'initial', 'display', 'block', 'margintop', '1em', 'marginbottom', '1em', 'textindent', '06em', 'pgfooter', 'divsecthead', 'fontsize', '110', 'fontweight', 'bold', 'pgfooter', 'projectgutenberglicense', 'fontsize', '110', 'margintop', '0', 'marginbottom', '0', 'textalign', 'center', 'pgheaderheading', 'inherit', 'textalign', 'center', 'fontsize', '110', 'pgfooterheading', 'inherit', 'textalign', 'center', 'fontsize', '120', 'fontweight', 'normal', 'margintop', '0', 'marginbottom', '0', 'pgheader', 'pgmachineheader', 'p', 'textindent', '4em', 'paddingleft', '4em', 'margintop', '1em', 'pgheader', 'pgheaderauthlist', 'p', 'marginleft', '2em', 'margintop', '0', 'marg

In [20]:
# Step 2: Splitting into training and validation sets
train_data, val_data = train_test_split(preprocessed_text, test_size=0.2, random_state=42)


In [25]:
train_data

['past',
 'valancyanything',
 'stickles',
 'nice',
 'one',
 'initiative',
 'skated',
 'p',
 'faces',
 'asked',
 'dad',
 'mr',
 'single',
 'stood',
 'came',
 'little',
 'fashion',
 'looked',
 'died',
 'bows',
 'drunk',
 'remarked',
 'im',
 'glass',
 'call',
 'project',
 'little',
 'dont',
 'p',
 'much',
 'outlaws',
 'ihimi',
 'waved',
 'valancy',
 'ribbons',
 'said',
 'snuffbrown',
 'p',
 'queer',
 'p',
 'stallinghad',
 'much',
 'lustreless',
 'span',
 'classdropcapsspanummer',
 'gays',
 'p',
 'world',
 'reputation',
 'spots',
 'mistclad',
 'last',
 'behind',
 'manner',
 'postmark',
 'given',
 'youre',
 'marblewhite',
 'fat',
 'company',
 'marginright',
 'heavily',
 'seen',
 'p',
 'fear',
 'go',
 'stirlings',
 'without',
 'couldnt',
 'willi',
 'silence',
 'barney',
 'heard',
 'longed',
 'could',
 'isabels',
 'jam',
 'p',
 'dimples',
 'way',
 'enough',
 'hale',
 'say',
 'convenient',
 'little',
 'crackle',
 'butof',
 'p',
 'earthly',
 'two',
 'unthinkable',
 'come',
 'id',
 'floor',
 'to

In [26]:
val_data

['deep',
 'afterlight',
 'castle',
 'yet',
 'looking',
 'p',
 'barney',
 'foundation',
 'sniffed',
 'loved',
 'remember',
 'p',
 'little',
 'matter',
 'pine',
 'utterly',
 'meals',
 'cut',
 'leander',
 'blue',
 'put',
 'little',
 'talked',
 'legs',
 'hrefchapter_xliv',
 'feel',
 'spree',
 'house',
 'damage',
 'john',
 'youd',
 'p',
 'see',
 'baby',
 'barney',
 'commonplaces',
 'dont',
 'bad',
 'purple',
 'function',
 'went',
 'troubles',
 'say',
 'straight',
 'excuse',
 'valancy',
 'island',
 'saluting',
 'hand',
 'implication',
 'classpginternalchapter',
 'pdiv',
 'superfluity',
 'doss',
 'stickles',
 'terrible',
 'ever',
 'thought',
 'mind',
 'classpginternalchapter',
 'evenings',
 'rubbers',
 'rose',
 'p',
 'dropped',
 'leaned',
 'told',
 'stand',
 'old',
 'xxviah2',
 'woodsa',
 'away',
 'bad',
 'either',
 'barney',
 'montrealin',
 'burden',
 'p',
 'commandi',
 'full',
 'p',
 'enough',
 'case',
 'thing',
 'impossible',
 'poisoned',
 'rumour',
 'phrase',
 'luck',
 'p',
 'whole',
 'si

In [27]:
# Step 3: Define your model architecture
class TextGenerator(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, num_heads, dropout):
        super(TextGenerator, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        assert embedding_dim % num_heads == 0  # Ensure embed_dim is divisible by num_heads
        self.transformer = nn.Transformer(
            d_model=embedding_dim,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=hidden_dim,
            dropout=dropout
        )
        self.fc = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output = self.transformer(embedded, embedded)
        output = self.fc(output)
        return output

In [28]:
# Step 4: Prepare your data for training
class TextDataset(Dataset):
    def __init__(self, data, vocab):
        self.data = data
        self.vocab = vocab

    def __getitem__(self, index):
        tokens = word_tokenize(self.data[index])
        indexed_tokens = [self.vocab.index(token) for token in tokens]
        return torch.tensor(indexed_tokens, dtype=torch.long)

    def __len__(self):
        return len(self.data)

vocab = list(set(train_data))
vocab_size = len(vocab)
embedding_dim = 100
hidden_dim = 256
num_layers = 4
num_heads = 4  # Change the number of heads to 4
dropout = 0.1

model = TextGenerator(vocab_size, embedding_dim, hidden_dim, num_layers, num_heads, dropout)


In [29]:
# Step 5: Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [30]:
# Step 6: Train your model
num_epochs = 10
batch_size = 32

train_dataset = TextDataset(train_data, vocab)
val_dataset = TextDataset(val_data, vocab)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [33]:
def evaluate_model(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            outputs = model(inputs)
            loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
            total_loss += loss.item()

    return total_loss / len(data_loader)
